# 1D Non-autonomous ODE

In [1]:
# import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import os
from sklearn.model_selection import train_test_split
import seaborn as sns
sns.set_style("whitegrid")
from model import *
from torch.utils.data import DataLoader, Dataset

torch.set_default_dtype(torch.float64)

## Read Data and Process Data

In [2]:
#Read data from .csv file
approx_sol = pd.read_csv('eg1_sol.csv', sep=',')
approx_solut = approx_sol.to_numpy()

#Data properties
components = 1 #spatial dimensions
deltat = 1/50 
t = np.arange(0.1, 2, deltat) #time steps of solution
time_steps = len(t)
n_initial_cond = approx_solut.shape[0] #number of initial conditions for solutions

#Read True RHS from .csv file
true_der = pd.read_csv('eg1_true_RHS.csv', sep=',')
true_der = true_der.to_numpy()

#Read Difference Quotients from .csv file
Diff_quot = pd.read_csv('eg1_app_RHS.csv', sep=',')
Diff_quot = Diff_quot.to_numpy()

In [3]:
# Append t and x(t)
t_copy = np.tile(t, n_initial_cond).reshape(-1,1)
x_t = approx_solut.reshape(-1,1) 
inputs = np.append(t_copy, x_t, axis=1)
targets = Diff_quot.reshape(-1,1)

# train-test split 
X_train, X_test, Y_train, Y_test = train_test_split(inputs, targets, test_size=0.2)

# visualization
#fontsize=15
#plt.scatter(X_train[:,0], X_train[:,1], color='skyblue', s=1)
#plt.scatter(X_test[:,0], X_test[:,1], color='orange', s=1)
#plt.xticks([0.5,1,1.5,2], size = fontsize, fontname = 'serif')
#plt.yticks([-1,1,3,5], size = fontsize, fontname = 'serif')
#plt.show()

In [4]:
#Define LoadDataset
class LoadDataset(Dataset):
    def __init__(self, inputs, targets):
        self.x = inputs
        self.y = targets

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        inputs = self.x[idx, :]
        targets = self.y[idx, :]

        return inputs, targets

## Generate non-trajectory data

In [5]:
n = 100 #grid points where we do the recovery

# Define real RHS
def z_func(t,x):
    return np.log(t) * np.exp(-x) - t ** 2

#define recovery domain boundaries
x_low = 0
x_top = 5
t_low = 0.1
t_top = 2.

#generate grid of x and t in the recovery domain
x = np.linspace(x_low, x_top, n)
time1 = np.linspace(t_low, t_top, n)
T,X = np.meshgrid(time1,x)

#compute true RHS on grid
Z = z_func(T, X)

#Generate input for netowrk
new_X = X.reshape(-1,1)
new_T = T.reshape(-1,1)
X_new = np.append(new_T,new_X,axis = 1)
net_input = torch.from_numpy(X_new)

## Define and train Lipschitz Neural Network

In [6]:
epochs = 15
batch_size = 100
input_size = 2 #columns in input matrix point
hidden_size = 64
output_size = 1 

# Network
gamma = 3
Lipschitz_network = LipNN(input_size,hidden_size,hidden_size, output_size, gamma)

#load dataset to use when training 
dataset = LoadDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train))
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True) 
#select Optimizer and parameters
optimizer = torch.optim.Adam(Lipschitz_network.parameters(), lr=1e-2, betas=(0.9, 0.999), eps=1e-08, weight_decay=0.0, amsgrad=False) 

loss = torch.nn.MSELoss()

for epoch in range(epochs):
    
    for input_data, targets in dataloader:
        preds = Lipschitz_network(input_data) 
        l2_loss = loss(preds, targets)
        
        #optimizer step
        optimizer.zero_grad()
        l2_loss.backward()
        optimizer.step()

In [7]:
print('Lipschitz Neural Network: ')

# Compute training loss
y_pred = Lipschitz_network(torch.from_numpy(X_train))
Lipschitz_train_loss = loss(y_pred, torch.from_numpy(Y_train))
print('Training loss is {:.2e}'.format(Lipschitz_train_loss))

y_pred = Lipschitz_network(torch.from_numpy(X_test))
Lipschitz_test_loss = loss(y_pred, torch.from_numpy(Y_test))
print('Test loss is {:.2e}'.format(Lipschitz_test_loss))

# generalization gap
Lip_gap = Lipschitz_test_loss - Lipschitz_train_loss
print('Generalization gap is {:.2e}'.format(Lip_gap))

# Lipschitz Neural Network
Z_lip = Lipschitz_network(net_input).detach().numpy()
Z_lip = Z_lip.reshape(n,n)
print('Mean Relative Error on non-trajectory data {0:1.2f}%'.format(100*np.mean(abs(Z - Z_lip))/(np.abs(np.max(Z)-np.min(Z)))))


Lipschitz Neural Network: 
Training loss is 8.95e-03
Test loss is 9.22e-03
Generalization gap is 2.66e-04
Mean Relative Error on non-trajectory data 2.03%


## Define and train Neural Network with Lip-regularization

In [8]:
epochs = 5
batch_size = 100
input_size = 2 #columns in input matrix point
hidden_size1 = 10
hidden_size2 = 30
output_size = 1 
reg_param = 0.005

how_many_points = 10

# Neural Network with Lip-regulariation term
Lip_reg = LipNet(input_size,hidden_size1,hidden_size2, output_size)

#load dataset to use when training 
dataset = LoadDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train))
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True) 
#select Optimizer and parameters
optimizer = torch.optim.Adam(Lip_reg.parameters(), lr=1e-2, betas=(0.9, 0.999), eps=1e-08, weight_decay=0.0, amsgrad=False) 

loss = torch.nn.MSELoss()

for epoch in range(epochs):
    
    for input_data, targets in dataloader:
        preds = Lip_reg(input_data) 
        l2_loss = loss(preds, targets)
        Lips =  reg_param * Lip(input_data, how_many_points, Lip_reg)
        lo = l2_loss +  Lips
        
        #optimizer step
        optimizer.zero_grad()
        lo.backward()
        optimizer.step()

In [9]:
print('Lipschitz Regularized Neural Network')

# Compute training loss
y_pred = Lip_reg(torch.from_numpy(X_train))
Lip_reg_train_loss = loss(y_pred, torch.from_numpy(Y_train))
print('Training loss is {:.2e}'.format(Lip_reg_train_loss))

y_pred = Lip_reg(torch.from_numpy(X_test))
Lip_reg_test_loss = loss(y_pred, torch.from_numpy(Y_test))
print('Test loss is {:.2e}'.format(Lip_reg_test_loss))

# generalization gap
Lip_reg_gap = Lip_reg_test_loss - Lip_reg_train_loss
print('Generalization gap is {:.2e}'.format(Lip_reg_gap))

# Neural Network with Lip regularization
Z_reg = Lip_reg(net_input).detach().numpy()
Z_reg = Z_reg.reshape(n,n)
print('Mean Relative Error on non-trajectory data is {0:1.2f}%'.format(100*np.mean(abs(Z - Z_reg))/(np.abs(np.max(Z)-np.min(Z)))))

Lipschitz Regularized Neural Network
Training loss is 7.99e-04
Test loss is 7.95e-04
Generalization gap is -3.23e-06
Mean Relative Error on non-trajectory data is 0.64%


## Define Random Features

In [6]:
############## Model 1: regularization parameter is 0.0001

T = 50    # repeat random sampling 10 times

N = 50
sigma = 1
reg_param = 0.0001

RF_train = np.zeros(T)
RF_test = np.zeros(T)
RF_gap = np.zeros(T)
RF_rel = np.zeros(T)

for t in range(T):
    
    # generate random features
    W = np.random.randn(2,N) / sigma
    b = np.random.uniform(0, 2*np.pi, size=(N,))
    
    # compute random features matrix and coefficients
    A = np.cos(X_train @ W + b) * np.sqrt(2/N)
    c = np.linalg.solve(A.T@A + reg_param * np.diag(np.linalg.norm(W,axis=0)), A.T@Y_train)

    # training loss
    y_pred = A @ c
    RF_train[t] = np.mean((y_pred - Y_train)**2)
    
    # test loss
    A_test = np.cos(X_test @ W + b) * np.sqrt(2/N)
    y_test = A_test @ c
    RF_test[t] = np.mean((y_test - Y_test)**2)

    # generalization gap
    RF_gap[t] = RF_test[t] - RF_train[t]
    
    # compute mean-relative error on non-trajectory data
    A = np.cos(X_new @ W + b) * np.sqrt(2/N)
    Z_RF = A @ c
    Z_RF = Z_RF.reshape(n,n)
    RF_rel[t] = np.mean(abs(Z - Z_RF))/ np.abs(np.max(Z)-np.min(Z))
    

print('Random Features:')
print(f'Training loss is {np.mean(RF_train):.2e}')
print(f'Test loss is {np.mean(RF_test):.2e}')
print(f'Generalization gap is {np.mean(RF_gap):.2e}')
print('Mean Relative Error on non-trajectory data is {0:1.2f}%'.format(100*np.mean(RF_rel)))

Random Features:
Training loss is 3.92e-05
Test loss is 3.95e-05
Generalization gap is 2.49e-07
Mean Relative Error on non-trajectory data is 0.67%


In [7]:
############## Model 1: regularization parameter is 0.0001

T = 50    # repeat random sampling 10 times

N = 50
sigma = 1
reg_param = 0.001

RF_train = np.zeros(T)
RF_test = np.zeros(T)
RF_gap = np.zeros(T)
RF_rel = np.zeros(T)

for t in range(T):
    
    # generate random features
    W = np.random.randn(2,N) / sigma
    b = np.random.uniform(0, 2*np.pi, size=(N,))
    
    # compute random features matrix and coefficients
    A = np.cos(X_train @ W + b) * np.sqrt(2/N)
    c = np.linalg.solve(A.T@A + reg_param * np.diag(np.linalg.norm(W,axis=0)), A.T@Y_train)

    # training loss
    y_pred = A @ c
    RF_train[t] = np.mean((y_pred - Y_train)**2)
    
    # test loss
    A_test = np.cos(X_test @ W + b) * np.sqrt(2/N)
    y_test = A_test @ c
    RF_test[t] = np.mean((y_test - Y_test)**2)

    # generalization gap
    RF_gap[t] = RF_test[t] - RF_train[t]
    
    # compute mean-relative error on non-trajectory data
    A = np.cos(X_new @ W + b) * np.sqrt(2/N)
    Z_RF = A @ c
    Z_RF = Z_RF.reshape(n,n)
    RF_rel[t] = np.mean(abs(Z - Z_RF))/ np.abs(np.max(Z)-np.min(Z))
    

print('Random Features:')
print(f'Training loss is {np.mean(RF_train):.2e}')
print(f'Test loss is {np.mean(RF_test):.2e}')
print(f'Generalization gap is {np.mean(RF_gap):.2e}')
print('Mean Relative Error on non-trajectory data is {0:1.2f}%'.format(100*np.mean(RF_rel)))

Random Features:
Training loss is 7.39e-05
Test loss is 7.58e-05
Generalization gap is 1.86e-06
Mean Relative Error on non-trajectory data is 0.79%


In [8]:
############## Model 1: regularization parameter is 0.0001

T = 50    # repeat random sampling 10 times

N = 50
sigma = 1
reg_param = 0.01

RF_train = np.zeros(T)
RF_test = np.zeros(T)
RF_gap = np.zeros(T)
RF_rel = np.zeros(T)

for t in range(T):
    
    # generate random features
    W = np.random.randn(2,N) / sigma
    b = np.random.uniform(0, 2*np.pi, size=(N,))
    
    # compute random features matrix and coefficients
    A = np.cos(X_train @ W + b) * np.sqrt(2/N)
    c = np.linalg.solve(A.T@A + reg_param * np.diag(np.linalg.norm(W,axis=0)), A.T@Y_train)
    
    # training loss
    y_pred = A @ c
    RF_train[t] = np.mean((y_pred - Y_train)**2)
    
    # test loss
    A_test = np.cos(X_test @ W + b) * np.sqrt(2/N)
    y_test = A_test @ c
    RF_test[t] = np.mean((y_test - Y_test)**2)

    # generalization gap
    RF_gap[t] = RF_test[t] - RF_train[t]
    
    # compute mean-relative error on non-trajectory data
    A = np.cos(X_new @ W + b) * np.sqrt(2/N)
    Z_RF = A @ c
    Z_RF = Z_RF.reshape(n,n)
    RF_rel[t] = np.mean(abs(Z - Z_RF))/ np.abs(np.max(Z)-np.min(Z))
    

print('Random Features:')
print(f'Training loss is {np.mean(RF_train):.2e}')
print(f'Test loss is {np.mean(RF_test):.2e}')
print(f'Generalization gap is {np.mean(RF_gap):.2e}')
print('Mean Relative Error on non-trajectory data is {0:1.2f}%'.format(100*np.mean(RF_rel)))

Random Features:
Training loss is 1.72e-04
Test loss is 1.77e-04
Generalization gap is 5.52e-06
Mean Relative Error on non-trajectory data is 0.95%


In [9]:
############## Model 1: regularization parameter is 0.0001

T = 50    # repeat random sampling 10 times

N = 50
sigma = 5
reg_param = 0.1

RF_train = np.zeros(T)
RF_test = np.zeros(T)
RF_gap = np.zeros(T)
RF_rel = np.zeros(T)

for t in range(T):
    
    # generate random features
    W = np.random.randn(2,N) / sigma
    b = np.random.uniform(0, 2*np.pi, size=(N,))
    
    # compute random features matrix and coefficients
    A = np.cos(X_train @ W + b) * np.sqrt(2/N)
    c = np.linalg.solve(A.T@A + reg_param * np.diag(np.linalg.norm(W,axis=0)), A.T@Y_train)

    # training loss
    y_pred = A @ c
    RF_train[t] = np.mean((y_pred - Y_train)**2)
    
    # test loss
    A_test = np.cos(X_test @ W + b) * np.sqrt(2/N)
    y_test = A_test @ c
    RF_test[t] = np.mean((y_test - Y_test)**2)

    # generalization gap
    RF_gap[t] = RF_test[t] - RF_train[t]
    
    # compute mean-relative error on non-trajectory data
    A = np.cos(X_new @ W + b) * np.sqrt(2/N)
    Z_RF = A @ c
    Z_RF = Z_RF.reshape(n,n)
    RF_rel[t] = np.mean(abs(Z - Z_RF))/ np.abs(np.max(Z)-np.min(Z))
    

print('Random Features:')
print(f'Training loss is {np.mean(RF_train):.2e}')
print(f'Test loss is {np.mean(RF_test):.2e}')
print(f'Generalization gap is {np.mean(RF_gap):.2e}')
print('Mean Relative Error on non-trajectory data is {0:1.2f}%'.format(100*np.mean(RF_rel)))

Random Features:
Training loss is 5.08e-03
Test loss is 5.31e-03
Generalization gap is 2.32e-04
Mean Relative Error on non-trajectory data is 1.56%
